# Agent Control Plane | Agent Infrastructure

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from typing import Dict, List, Optional
from dataclasses import dataclass, field
from datetime import datetime
import time

set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [2]:
model = ChatOpenAI(model="gpt-4o")

In [3]:
# Agent Control Plane Implementation

@dataclass
class AgentPolicy:
    max_requests_per_minute: int = 60
    max_tokens_per_request: int = 4000
    allowed_tools: List[str] = field(default_factory=list)
    require_human_approval: bool = False

@dataclass
class AgentMetrics:
    total_requests: int = 0
    total_errors: int = 0
    avg_latency_ms: float = 0.0
    last_active: Optional[str] = None

class ControlPlane:
    """Centralized control plane for managing agents."""

    def __init__(self):
        self._agents: Dict[str, dict] = {}
        self._policies: Dict[str, AgentPolicy] = {}
        self._metrics: Dict[str, AgentMetrics] = {}
        self._rate_counters: Dict[str, List[float]] = {}

    def deploy_agent(self, name: str, version: str, policy: Optional[AgentPolicy] = None):
        """Deploy or update an agent."""
        self._agents[name] = {
            "version": version,
            "status": "running",
            "deployed_at": datetime.now().isoformat(),
        }
        self._policies[name] = policy or AgentPolicy()
        self._metrics[name] = AgentMetrics()
        self._rate_counters[name] = []
        return f"Deployed {name} v{version}"

    def check_rate_limit(self, name: str) -> bool:
        """Check if agent is within rate limits."""
        policy = self._policies.get(name)
        if not policy:
            return False
        now = time.time()
        window = [t for t in self._rate_counters.get(name, []) if now - t < 60]
        self._rate_counters[name] = window
        if len(window) >= policy.max_requests_per_minute:
            return False
        self._rate_counters[name].append(now)
        return True

    def invoke_agent(self, name: str, prompt: str) -> str:
        """Invoke an agent through the control plane with policy enforcement."""
        agent = self._agents.get(name)
        if not agent or agent["status"] != "running":
            return f"Error: Agent '{name}' is not available (status: {agent.get('status', 'not found') if agent else 'not found'})"

        if not self.check_rate_limit(name):
            self._metrics[name].total_errors += 1
            return f"Error: Rate limit exceeded for '{name}'"

        start = time.time()
        try:
            response = model.invoke(prompt)
            latency = (time.time() - start) * 1000
            metrics = self._metrics[name]
            metrics.total_requests += 1
            metrics.avg_latency_ms = (
                (metrics.avg_latency_ms * (metrics.total_requests - 1) + latency)
                / metrics.total_requests
            )
            metrics.last_active = datetime.now().isoformat()
            return response.content
        except Exception as e:
            self._metrics[name].total_errors += 1
            return f"Error: {str(e)}"

    def get_dashboard(self) -> str:
        """Get a dashboard view of all agents."""
        lines = ["AGENT CONTROL PLANE DASHBOARD", "=" * 50]
        for name, agent in self._agents.items():
            metrics = self._metrics.get(name, AgentMetrics())
            policy = self._policies.get(name, AgentPolicy())
            lines.append(
                f"\n{name} (v{agent['version']}) - {agent['status']}\n"
                f"  Requests: {metrics.total_requests} | Errors: {metrics.total_errors} | "
                f"Avg Latency: {metrics.avg_latency_ms:.0f}ms\n"
                f"  Rate Limit: {policy.max_requests_per_minute}/min | "
                f"Human Approval: {policy.require_human_approval}\n"
                f"  Last Active: {metrics.last_active or 'never'}"
            )
        return "\n".join(lines)

    def stop_agent(self, name: str) -> str:
        if name in self._agents:
            self._agents[name]["status"] = "stopped"
            return f"Stopped {name}"
        return f"Agent '{name}' not found"

In [4]:
# Usage
cp = ControlPlane()

# Deploy agents with policies
cp.deploy_agent("analyzer", "1.0", AgentPolicy(max_requests_per_minute=30))
cp.deploy_agent("writer", "2.1", AgentPolicy(max_requests_per_minute=20, require_human_approval=True))

# Invoke through control plane
result = cp.invoke_agent("analyzer", "Analyze the pros and cons of microservices architecture")
print(f"Analyzer says: {result[:200]}...")

result = cp.invoke_agent("writer", "Write a brief memo about adopting AI agents in enterprise")
print(f"\nWriter says: {result[:200]}...")

# Dashboard
print(f"\n{cp.get_dashboard()}")

Analyzer says: Microservices architecture is a software design pattern that structures an application as a collection of loosely coupled services, each responsible for a specific business capability. This architectu...

Writer says: ---
**MEMO**

**To:** Executive Team  
**From:** [Your Name]  
**Date:** [Current Date]  
**Subject:** Adoption of AI Agents in Our Enterprise

---

**Introduction**

As technological advancements con...

AGENT CONTROL PLANE DASHBOARD

analyzer (v1.0) - running
  Requests: 1 | Errors: 0 | Avg Latency: 7ms
  Rate Limit: 30/min | Human Approval: False
  Last Active: 2026-04-09T11:28:39.873584

writer (v2.1) - running
  Requests: 1 | Errors: 0 | Avg Latency: 0ms
  Rate Limit: 20/min | Human Approval: True
  Last Active: 2026-04-09T11:28:39.874146
